# XYZ — Breast Cancer Wisconsin

Exemplo da API `fit()/transform()` com **6 features fixas** e todas as 569 amostras do Breast Cancer Wisconsin. O alvo positivo é tumor maligno.

As colunas são definidas previamente por nome. Com `q_enc=2` e três eixos, o XYZ codifica as seis features em um bloco multieixo. O Hamiltoniano é apresentado em [Generalized two-qubit Hamiltonian for Projective Quantum Feature Maps](https://arxiv.org/abs/2606.13641).

O PQFM permanece dentro do pipeline para impedir que mutual information, blocks e feature assignment do fold de validação entrem no `fit()`.

O notebook faz uma comparação direta entre um modelo clássico treinado com as features originais e um modelo treinado somente com as features quânticas produzidas pelo PQFM.

In [1]:
from __future__ import annotations

from getpass import getpass
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
IDEAL = True
SIMULATION = True
SHOTS = 4096
IBM_BACKEND_NAME = "ibm_kingston"

from sklearn.datasets import load_breast_cancer

from pqfmlib import XYZProjectiveQFM

## Credenciais IBM para `ideal=False`

O modo padrão é `IDEAL = True` e não solicita credenciais. Para testar `ideal=False`, altere essa variável na primeira célula.

A célula seguinte solicitará o token sem exibi-lo e o disponibilizará aos clones pela variável de ambiente `QISKIT_IBM_TOKEN`. O token não será salvo em arquivo pelo notebook. O único backend carregado será `ibm_kingston`.

A execução continua com `simulation=True` e `fakebackend=True`: as propriedades do Kingston são usadas na simulação, mas nenhum job é submetido à QPU.

In [2]:
if not IDEAL:
    print("O token é usado somente nesta sessão e não será salvo pelo notebook.")
    os.environ["QISKIT_IBM_TOKEN"] = getpass("IBM Cloud API token: ")

In [3]:
dataset = load_breast_cancer(as_frame=True)
selected_features = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness",
    "mean compactness",
]
X = dataset.data.loc[:, selected_features].copy()
y = (dataset.target.to_numpy() == 0).astype(int)  # maligno = classe positiva
X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness
0,17.99,10.38,122.80,1001.0,0.11840,0.27760
1,20.57,17.77,132.90,1326.0,0.08474,0.07864
2,19.69,21.25,130.00,1203.0,0.10960,0.15990
3,11.42,20.38,77.58,386.1,0.14250,0.28390
4,20.29,14.34,135.10,1297.0,0.10030,0.13280


In [4]:
pqfm = XYZProjectiveQFM(
    name_file="breast_cancer_grid_search",
    seed=SEED,
    ideal=IDEAL,
    simulation=SIMULATION,
    fakebackend=not IDEAL,
    shots=SHOTS,
    ibm_qpu=IBM_BACKEND_NAME,
    q_enc=2,
    features_per_qubit=3,
    axes=("x", "y", "z"),
    encoding_mode="multi_axis",
    m=1,
    keep_diagonal_terms=True,
    keep_cross_terms=True,
    use_gpu_statevector=False,
)

## Comparação com GridSearchCV nos 80% de treino

Dois modelos são ajustados com os mesmos cinco folds e a mesma grade de hiperparâmetros:

- **Clássico:** features originais escaladas → GradientBoosting.
- **PQFM:** features originais escaladas → PQFM → GradientBoosting.

O critério de seleção é exclusivamente ROC-AUC. O threshold final permanece no valor padrão `0.5`.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

classical_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("classifier", GradientBoostingClassifier(random_state=SEED)),
    ]
)

quantum_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("pqfm", pqfm),
        ("classifier", GradientBoostingClassifier(random_state=SEED)),
    ]
)

parameter_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__learning_rate": [0.05],
    "classifier__max_depth": [1, 2],
}


def make_grid_search(pipeline):
    return GridSearchCV(
        estimator=pipeline,
        param_grid=parameter_grid,
        scoring="roc_auc",
        cv=cv,
        refit=True,
        n_jobs=1,
        return_train_score=False,
        verbose=1,
    )


classical_search = make_grid_search(classical_pipeline)
quantum_search = make_grid_search(quantum_pipeline)

classical_search.fit(X_train, y_train)
quantum_search.fit(X_train, y_train)

cv_comparison = pd.DataFrame(
    {
        "Modelo": ["Clássico", "PQFM"],
        "Melhor ROC-AUC CV": [
            classical_search.best_score_,
            quantum_search.best_score_,
        ],
        "Melhores hiperparâmetros": [
            classical_search.best_params_,
            quantum_search.best_params_,
        ],
    }
).set_index("Modelo")

print(f"Dataset completo: {len(X)} amostras")
print(f"Treino/GridSearchCV: {len(X_train)} amostras")
print(f"Teste final: {len(X_test)} amostras")
display(cv_comparison)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Dataset completo: 569 amostras
Treino/GridSearchCV: 455 amostras
Teste final: 114 amostras


,Melhor ROC-AUC CV,Melhores hiperparâmetros
Modelo,,
Clássico,0.979979,"{'classifier__learning_rate': 0.05, 'classifie..."
PQFM,0.834004,"{'classifier__learning_rate': 0.05, 'classifie..."


## Comparação final nos 20% de teste

Os dois melhores estimadores são avaliados uma única vez no mesmo conjunto de teste, que não participa da escolha dos hiperparâmetros.

In [6]:
def evaluate_on_test(search):
    probabilities = search.predict_proba(X_test)[:, 1]
    predictions = search.predict(X_test)
    metrics = {
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0),
        "Accuracy": accuracy_score(y_test, predictions),
    }
    return metrics, probabilities, predictions


classical_metrics, classical_probabilities, classical_predictions = evaluate_on_test(
    classical_search
)
quantum_metrics, quantum_probabilities, quantum_predictions = evaluate_on_test(
    quantum_search
)

test_comparison = pd.DataFrame(
    {
        "Clássico": classical_metrics,
        "PQFM": quantum_metrics,
    }
).T
test_comparison.index.name = "Modelo"
display(test_comparison)

best_pqfm = quantum_search.best_estimator_.named_steps["pqfm"]
scaled_example = quantum_search.best_estimator_.named_steps["scaler"].transform(
    X_test.iloc[:1]
)
n_quantum_features = best_pqfm.transform(scaled_example).shape[1]
print("Número de features clássicas:", X.shape[1])
print("Número de features quânticas:", n_quantum_features)
print("Nós físicos do melhor PQFM:", getattr(best_pqfm, "phys_nodes", None))

,ROC-AUC,Recall,Precision,F1-score,Accuracy
Modelo,,,,,
Clássico,0.976521,0.833333,0.875000,0.853659,0.894737
PQFM,0.854828,0.500000,0.677419,0.575342,0.728070


Número de features clássicas: 6
Número de features quânticas: 9
Nós físicos do melhor PQFM: [0, 1]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

RocCurveDisplay.from_predictions(
    y_test,
    classical_probabilities,
    name="Clássico",
    ax=axes[0],
)
RocCurveDisplay.from_predictions(
    y_test,
    quantum_probabilities,
    name="PQFM",
    ax=axes[0],
)
axes[0].set_title("ROC no teste final")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    classical_predictions,
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title("Clássico (threshold = 0.5)")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    quantum_predictions,
    ax=axes[2],
    colorbar=False,
)
axes[2].set_title("PQFM (threshold = 0.5)")

plt.tight_layout()
plt.show()